In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('bmh')
plt.rcParams['axes.facecolor'] = 'white'

In [2]:
y_pred_high = np.load('../lstm-anom-y_pred_high.npy', allow_pickle=True)
y_pred_low = np.load('../lstm-anom-y_pred_low.npy', allow_pickle=True)
y = np.load('../lstm-anom-y.npy', allow_pickle=True)

In [3]:
np.mean(
    (y_pred_low[:, :, 0] < y[:, :, 0]) &
    (y_pred_high[:, :, 0] > y[:, :, 0])
)

np.float64(0.8284393063583815)

In [4]:
np.mean(
    (y_pred_low[:, :, 1] < y[:, :, 1]) &
    (y_pred_high[:, :, 1] > y[:, :, 1])
)

np.float64(0.7817630057803469)

In [5]:
from datetime import datetime
timestamps = list(pd.date_range(start=datetime(2024,11,18,0,0), end=datetime(2024,11,19,0,0), freq='60s')[31:-26])

In [6]:
y_true = np.load('../val-set-y_true.npy', allow_pickle=True)
y_low = np.load('../val-set-y_pred_low.npy', allow_pickle=True)
y_high = np.load('../val-set-y_pred_high.npy', allow_pickle=True)

In [7]:
np.mean( (y_true < y_high) & (y_true > y_low), axis=0 ).mean(axis=0)

array([0.93247236, 0.94765788])

In [8]:
np.mean( np.abs(y_high - y_low), axis=0).mean(axis=0)

array([0.2736201 , 0.14062777], dtype=float32)

In [9]:
in_pi_mask = (y_true < y_high) & (y_true > y_low)
val_mmsis = np.load('../lstm-val-mmsis-test.npy', allow_pickle=True)
val_days = np.load('../lstm-val-days-test.npy', allow_pickle=True)

In [10]:
group_results = {}
for ix, (mmsi, day) in enumerate(zip(val_mmsis, val_days)):
    
    if (mmsi, day) not in group_results:
        group_results[(mmsi, day)] = [ in_pi_mask[ix] ]
    else:
        group_results[(mmsi, day)].append( in_pi_mask[ix] )

df = []

for (mmsi, day), pi_masks in group_results.items():

    df.append({
        'mmsi': mmsi,
        'day': day,
        'delta_cog': np.mean( np.mean(pi_masks, axis=0)[:, 0] ), 
        'delta_dif': np.mean( np.mean(pi_masks, axis=0)[:, 1] ),
    })

df = pd.DataFrame(df)
df

,mmsi,day,delta_cog,delta_dif
0,305922000,2024-12-06,0.962156,0.969341
1,305922000,2024-12-07,0.965854,0.991916
2,305943000,2024-11-15,0.997500,0.995000
3,305967000,2024-11-15,0.950358,0.988143
4,305967000,2024-11-18,0.788308,0.902769
...,...,...,...,...
315,518999083,2024-11-18,0.989091,1.000000
316,518999147,2024-12-12,0.959326,0.951236
317,525007403,2024-12-06,0.764651,0.867907
318,525007403,2024-12-07,0.956390,0.958829


In [11]:
thresh_cog = np.quantile( df['delta_cog'], 0.05 )
thresh_cog

np.float64(0.8329080852921913)

In [12]:
(df['delta_cog'] == 1.0).mean()

np.float64(0.046875)

In [13]:
(df['delta_dif'] == 1.0).mean()

np.float64(0.05)

In [14]:
thresh_dif = np.quantile( df['delta_dif'], 0.05 )
thresh_dif

np.float64(0.8916719576719576)

In [15]:
y_true = np.load('../test-set-y_true.npy', allow_pickle=True)
y_low = np.load('../test-set-y_pred_low.npy', allow_pickle=True)
y_high = np.load('../test-set-y_pred_high.npy', allow_pickle=True)

in_pi_mask = (y_true < y_high) & (y_true > y_low)

test_mmsis = np.load('../lstm-all-mmsis-test.npy', allow_pickle=True)
test_days = np.load('../lstm-all-days-test.npy', allow_pickle=True)

group_results = {}
for ix, (mmsi, day) in enumerate(zip(test_mmsis, test_days)):
    
    if (mmsi, day) not in group_results:
        group_results[(mmsi, day)] = [ in_pi_mask[ix] ]
    else:
        group_results[(mmsi, day)].append( in_pi_mask[ix] )

df = []

for (mmsi, day), pi_masks in group_results.items():

    df.append({
        'mmsi': mmsi,
        'day': day,
        'delta_cog': np.mean(pi_masks, axis=0)[0, 0],
        'delta_dif': np.mean(pi_masks, axis=0)[0, 1],
    })

df = pd.DataFrame(df)
print( np.mean( df['delta_dif'] < thresh_dif ) )
print( np.mean( df['delta_cog'] < thresh_cog ) )

0.052307692307692305
0.046153846153846156
